# Moon synthetics from the RSGT catalogue — Python

The catalogue holds, for each of 1000 lunar 1-D models, the **receiver-side strain Green's
tensors** (computed with flexOPT.jl, OPT3, 0–0.05 Hz, elastic). With reciprocity, they give the
seismograms at any station for any event (position, depth down to the top of the core, moment
tensor or force) in a fraction of a second, with no solver.

You need the folder of `rsgt_XXXXX.h5` files (one per model, ≈ 0.6 GB each) and, in the same
folder as this notebook, `moon_rsgt_py.py`.

Conventions: angles in degrees, depth in km, moment tensor in the GCMT convention
(r = up, t = south, p = east) in N·m, force (f_r, f_t, f_p) in N. Components Z (up), N, E,
R (horizontal, away from the event), T. Output: velocity (m/s) by default.

## 1. Where is the catalogue?

Change the path below. If files are still being copied, only use the ones that are complete.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import moon_rsgt_py as moon

moon.CATALOGUE = os.path.expanduser("~/Dropbox/stage_tutore_2026/catalogue_h5")
ids = moon.model_ids()
print(len(ids), "models:", ids[:5], "...")

## 2. What is inside one file?

In [ ]:
import h5py
mid = ids[0]
with h5py.File(moon.model_file(mid), "r") as f:
    for k in f.keys():
        d = f[k]
        print(f"{k:20s} {str(d.shape):22s} {d.dtype}")
    print("Moon radius", f["R_km"][()], "km; deepest event depth", f["max_depth_km"][()], "km")

## 3. One event, one station (Z, R, T velocity)

In [ ]:
station = dict(stla=-3.0, stlo=-23.4)                       # near Apollo 12
event = dict(evla=20.0, evlo=15.0, evdp=900.0)              # a deep moonquake
M = moon.double_couple(strike=30, dip=60, rake=90, m0=1e15)  # N.m

t, tr = moon.synthetics(mid, **station, **event, **M)        # velocity, m/s
print("distance", round(moon.epicentral_distance(event["evla"], event["evlo"], station["stla"], station["stlo"]), 2), "deg")
fig, axs = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
for ax, c in zip(axs, "ZRT"):
    ax.plot(t, tr[c], "k", lw=0.8); ax.set_ylabel(f"{c} (m/s)")
axs[-1].set_xlabel("time (s)"); axs[0].set_title(f"model {mid}")
plt.show()

## 4. Record section (stations every 10° along one azimuth)

In [ ]:
ev = dict(evla=0.0, evlo=0.0, evdp=300.0)
dists = np.arange(5, 180, 10)
plt.figure(figsize=(10, 8))
for dlt in dists:
    stla, stlo = moon.point_at(ev["evla"], ev["evlo"], dlt, 45.0)
    t, tr = moon.synthetics(mid, stla=stla, stlo=stlo, **ev, **M, comps="Z")
    z = tr["Z"] / np.abs(tr["Z"]).max()
    plt.plot(t, dlt + 4 * z, "k", lw=0.6)
plt.xlabel("time (s)"); plt.ylabel("epicentral distance (deg)")
plt.title(f"model {mid}, Z velocity, event at {ev['evdp']} km (each trace normalised)")
plt.show()

## 5. The same event and station for several models

In [ ]:
some = ids[:5]
plt.figure(figsize=(10, 4))
for m in some:
    t, tr = moon.synthetics(m, **station, **event, **M, comps="Z")
    plt.plot(t, tr["Z"], lw=0.8, label=m)
plt.legend(); plt.xlabel("time (s)"); plt.ylabel("Z velocity (m/s)")
plt.show()

## 6. An impact: vertical force at the surface

In [ ]:
t, tr = moon.synthetics(mid, **station, evla=5.0, evlo=40.0, evdp=0.0, force=(-1e10, 0.0, 0.0), comps="ZRT")
fig, axs = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
for ax, c in zip(axs, "ZRT"):
    ax.plot(t, tr[c], "k", lw=0.8); ax.set_ylabel(f"{c} (m/s)")
axs[-1].set_xlabel("time (s)")
plt.show()

Options: `comps="ZNE"`, `quantity="displacement"` or `"acceleration"`, `dt=0.5`, `half_duration=15.0` (s, keep ≥ 10 s: the catalogue stops at 0.05 Hz). `moon.max_depth(mid)` gives the deepest allowed event depth of a model.